<a href="https://colab.research.google.com/github/EmilioUcelayLojo/AA3/blob/laboratorios_practicas/lab8/lab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA3/blob/main/lab8/lab8.ipynb)

# Práctica 8: Modelos generativos

## Pre-requisitos

### Instalar paquetes

Si la práctica requiere algún paquete de Python, habrá que incluir una celda en la que se instalen. Si usamos un paquete que se ha utilizado en prácticas anteriores, podríamos dar por supuesto que está instalado pero no cuesta nada satisfacer todas las dependencias en la propia práctica para reducir las dependencias entre ellas.

### NOTA: En <font color='red'>Google Colab</font> hay que instalar los paquetes EN CADA EJECUCIÓN

In [12]:
# Ejemplo de instalación de tensorflow 2.0
#%tensorflow_version 2.x
# !pip3 install tensorflow  # NECESARIO SOLO SI SE EJECUTA EN LOCAL
import tensorflow as tf

# Hacemos los imports que sean necesarios
import numpy as np

# Modelos generativos sobre MNIST

Lo primero que tenemos que hacer es cargar el dataset.

In [13]:
labeled_data = 0.01 # Vamos a usar el etiquetado de sólo el 1% de los datos
np.random.seed(42)

(x_train, y_train), (x_test, y_test), = tf.keras.datasets.mnist.load_data()

indices = np.arange(len(x_train))
np.random.shuffle(indices)
ntrain_data = int(labeled_data*len(x_train))
unlabeled_train = x_train[indices[ntrain_data:]]
x_train = x_train[indices[:ntrain_data]]
y_train = y_train[indices[:ntrain_data]]

print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

x_train shape: (600, 28, 28)
600 train samples
10000 test samples


In [3]:
# TODO: Haz el preprocesado que necesites aquí (si lo necesitas)
# Normalizar los valores de los píxeles al rango [0, 1]
x_train = x_train.astype(np.float32) / 255.0
unlabeled_train = unlabeled_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# Aplanar las imágenes para que sean vectores de características
x_train = x_train.reshape(x_train.shape[0], -1)
unlabeled_train = unlabeled_train.reshape(unlabeled_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

## Modelo generativo

Vamos a crear nuestro propio modelo generativo. En clase de teoría has visto muchas versiones distintas:

1. Mezcla de distribuciones de Gaussianas (GMM)
1. Mezcla de distribuciones multinomiales (Naive Bayes)
1. Modelos de Markov ocultos (HMM)

Tal y como se os apunta en teoría, los modelos generativos abordan un problema más general que la clasificación o regresión: aprenden cómo se estructuran y distribuyen los datos de entrada.

En nuestro caso, vamos a modelar los datos de entrada mediante el uso de **Autoencoders**.

# Autoencoders

El autoencoder es un tipo de red que se utiliza para aprender codificaciones eficientes de datos sin etiquetar (lo que se conoce como aprendizaje no supervisado). Es una red que tiene el mismo tamaño en la entrada como en la salida, puesto que el objetivo de la red es reconstruir la entrada con la menor pérdida posible.

Si lo que hacemos es reconstruir la entrada, ¿qué sentido tiene el usar la red? Habitualmente, **la red consta, a su mitad, de una capa con menos elementos que los datos de entrada**. Por tanto, al reconstruir los datos de la entrada a la salida, en esa capa tendremos una versión *comprimida* de la entrada, que contendrá la mayor parte de su información.

Por tanto, podemos dividir un autoencoder en 3 secciones diferentes, tal y como se ve en la siguiente figura:

![](https://drive.google.com/uc?export=view&id=1yxkKZV0J0YplQAGPGJxQ2Z80Ad6L94eu)

1. **Encoder:** es la parte inicial de la red, encargada de comprimir los datos de la entrada.
1. **Code:** es la salida del encoder, contiene la versión *comprimida* de los datos de entrada.
1. **Decoder:** se encarga de, partiendo de la salida del *Encoder*, reconstruir la red.

## Crea tu propio Autoencoder

El diseño del autoencoder es libre (capas densas, convolucionales, ...), puedes crearlo como quieras. **El único requisito es que tiene que mantener los nombres (y parámetros) de las funciones descritas abajo.**

In [4]:
# TODO: crea tu propio autoencoder

class MiAutoencoder:

    def __init__(self, input_shape):
        # TODO : define el modelo y compílalo
        # Define la arquitectura del encoder
        self.encoder = tf.keras.models.Sequential([
            tf.keras.layers.Input(shape=input_shape),       # Entrada del encoder
            tf.keras.layers.Dense(128, activation='relu'),  # Capa oculta 1
            tf.keras.layers.Dense(64, activation='relu'),   # Capa oculta 2 (code)
        ])

        # Define la arquitectura del decoder
        self.decoder = tf.keras.models.Sequential([
            tf.keras.layers.Input(shape=(64,)),            # Entrada del decoder (code)
            tf.keras.layers.Dense(128, activation='relu'),  # Capa oculta 1
            tf.keras.layers.Dense(input_shape, activation='sigmoid'), # Capa de salida
        ])

        # Crea el modelo completo del autoencoder
        self.autoencoder = tf.keras.models.Model(inputs=self.encoder.input,
                                                outputs=self.decoder(self.encoder.output))

        # Compila el modelo
        self.autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

    def fit(self, X, y=None, sample_weight=None):
        # TODO: entrena el modelo. Escoge el tamaño de batch y el número de epochs que quieras
        # Entrena el autoencoder
        self.autoencoder.fit(X, X, epochs=10, batch_size=32)  # Ajusta epochs y batch_size según sea necesario

    def get_encoded_data(self, X):
        # TODO: devuelve la salida del encoder (code)
        # Obtiene la salida del encoder (code)
        return self.encoder.predict(X)

    def __del__(self):
        # elimina todos los modelos que hayas creado
        tf.keras.backend.clear_session() # Necesario para liberar la memoria en GPU

## Crea tu propio Clasificador

A continuación crearemos un clasificador que sea capaz de predecir la etiqueta pero no a partir de los datos originales, sino de la codificación `code` aprendida por el autoencoder. El diseño del clasificador es libre, pero recuerda que tiene que ser simple (máximo dos capas). **El único requisito es que tiene que mantener los nombres (y parámetros) de las funciones descritas abajo.**

In [5]:
# TODO: crea tu propio clasificador

class MiClasificador:

    def __init__(self):
        # TODO : define el modelo y compílalo
        # Define la arquitectura del clasificador
        self.classifier = tf.keras.models.Sequential([
            tf.keras.layers.Input(shape=(64,)),  # La entrada es la salida del encoder (code)
            tf.keras.layers.Dense(32, activation='relu'),  # Capa oculta
            tf.keras.layers.Dense(10, activation='softmax')  # Capa de salida (10 clases para MNIST)
        ])

        # Compila el modelo
        self.classifier.compile(optimizer='adam',
                               loss='sparse_categorical_crossentropy',
                               metrics=['accuracy'])

    def fit(self, X, y, sample_weight=None):
        # TODO: entrena el modelo. Escoge el tamaño de batch y el número de epochs que quieras
        # Entrena el clasificador
        self.classifier.fit(X, y, epochs=10, batch_size=32)  # Ajusta epochs y batch_size según sea necesario

    def predict(self, X):
        # TODO: devuelve la clase ganadora
        # Devuelve la clase con la probabilidad más alta
        return np.argmax(self.classifier.predict(X), axis=-1)

    def predict_proba(self, X):
        # Devuelve las probabilidades de cada clase
        return self.classifier.predict(X)

    def score(self, X, y):
        # Calcula la precisión del clasificador
        loss, accuracy = self.classifier.evaluate(X, y, verbose=0)
        return accuracy

    def __del__(self):
        # elimina todos los modelos que hayas creado
        tf.keras.backend.clear_session() # Necesario para liberar la memoria en GPU

### Entrenamiendo del modelo sin supervisar

Primero de todo, a modo de comparación, crea un modelo (de capacidad similar a tu encoder+clasificador) que puedas entrenar supervisadamente con `x_train` e `y_train`. Anota su rendimiento.

In [6]:
# TODO - Crea un modelo, entrénalo con x_train e y_train y muestra su rendimiento en test.

# Crea un modelo simple con dos capas densas
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compila el modelo
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Entrena el modelo
model.fit(x_train, y_train, epochs=10, batch_size=32)

# Evalúa el modelo en el conjunto de test
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print('Precisión en el conjunto de test:', accuracy)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3054 - loss: 2.0648
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7964 - loss: 1.0287
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8507 - loss: 0.6389
Epoch 4/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9422 - loss: 0.3912
Epoch 5/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9234 - loss: 0.3600
Epoch 6/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9605 - loss: 0.2508
Epoch 7/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9811 - loss: 0.1977
Epoch 8/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9787 - loss: 0.1854
Epoch 9/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9850 - loss: 0.1610
Epoch 10/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9951 - loss: 0.1159
Precisión en el conjunto de test: 0.8762000203132629


### Entrenamiendo del modelo semisupervisado

El entrenamiento del sistema semisupervisado se realiza en dos pasos.

1. Se entrena el autoencoder con todos los datos (etiquetados y sin etiquetar).
1. Se entrena un clasificador simple (una o dos capas), teniendo como entrada la salida del encoder (**code**) de los datos etiquetados.

<font color='red'>NOTA:</font> para entrenar (y predecir) vamos a utilizar las funciones que hemos definido en el autoencoder y en el clasificador.

In [7]:
# TODO: implementa el algoritmo semisupervised_training.

def semisupervised_training(autoencoder, classifier, x_train, y_train, unlabeled_data):
    """Entrena un modelo de forma semisupervisada.

    Args:
        autoencoder: El modelo de autoencoder.
        classifier: El modelo de clasificador.
        x_train: Los datos de entrenamiento etiquetados.
        y_train: Las etiquetas de los datos de entrenamiento etiquetados.
        unlabeled_data: Los datos de entrenamiento no etiquetados.
    """
    # 1. Entrena el autoencoder con todos los datos (etiquetados y no etiquetados).
    autoencoder.fit(np.concatenate([x_train, unlabeled_data]))

    # 2. Obtén la codificación (code) de los datos etiquetados.
    encoded_x_train = autoencoder.get_encoded_data(x_train)

    # 3. Entrena el clasificador con la codificación de los datos etiquetados.
    classifier.fit(encoded_x_train, y_train)

### Entrenamos nuestro modelo

Usa lo hecho anteriormente para entrenar tu clasificador de una manera semi-supervisada.

In [11]:
# Crea tu autoencoder y tu clasificador
autoencoder = MiAutoencoder(input_shape=x_train.shape[1],)  # Crea una instancia de MiAutoencoder
classifier = MiClasificador()  # Crea una instancia de MiClasificador

ValueError: Cannot convert '784' to a shape.

In [ ]:
# TODO: Entrena tu modelo
None

In [ ]:
# TODO: Obtén la precisión sobre el conjunto de test
pred_data = autoencoder.get_encoded_data(x_test)
print('Test accuracy :', classifier.score(pred_data, y_test))

## Mejorando el código

nuestro modelo actual requiere de dos pasos para entrenarse, pero podría realizarse en un único paso si **creamos un modelo con las dos salidas (autoencoder y clasificador)**.

Para ello, hay que tener en cuenta que, en los datos sin etiquetar, su contribución al clasificador debería ser nula.


### TRABAJO: Crea el nuevo modelo y modifica la función semisupervised_training para tener en cuenta todos los puntos mencionados anteriormente

In [ ]:
# TODO: crea el nuevo modelo

# TODO: crea tu propio clasificador

class MiClasificadorSemisupervisado:

    def __init__(self, input_shape):
        # TODO : define el modelo y compílalo
        None

    def fit(self, X, y, unlabeled_data):
        # TODO: entrena el modelo. Escoge el tamaño de batch y el número de epochs que quieras, y define bien el sample_weight
        None

    def predict(self, X):
        # TODO: devuelve la clase ganadora del clasificador
        None

    def predict_proba(self, X):
        # TODO: devuelve la probabilidad del clasificador
        None

    def score(self, X, y):
        None

    def __del__(self):
        # elimina todos los modelos que hayas creado
        tf.keras.backend.clear_session() # Necesario para liberar la memoria en GPU

In [ ]:
# TODO: reescribe la función semisupervised_training para incorporar las mejoras mencionadas anteriormente

def semisupervised_training_v2(model, x_train, y_train, unlabeled_data):
    None

In [ ]:
# TODO: Crea y entrena tu clasificador
model = None

In [ ]:
# TODO: Obtén la precisión sobre el conjunto de test
print('Test accuracy :', model.score(x_test, y_test))

# Hay vida más allá del autoencoder

¿Has probado a utilizar otro método distinto del autoencoder para obtener una respresentación similar a la salida del encoder? La idea es la siguiente:

1. Define un modelo $model$ convolucional similar al encoder de un autoencoder (la entrada es el tamaño de la imagen, la salida el vector de representación)
1. Define una capa de salida $cluster$ que, partiendo de la salida de model, nos devuelva una salida con el mismo número de clases que el dataset a utilizar (la entrada es el vector de representación), usando softmax como activación de salida
1. Para cada batch de entrenamiento $X$:  # Usa un batch alto, mínimo 128
  1. Modifica las imágenes de entrada con [data_augmentation](https://www.tensorflow.org/tutorials/images/data_augmentation?hl=es-419), llámala $augX_1$.
  1. Modifica otra vez las imágenes de entrada con [data_augmentation_2](https://www.tensorflow.org/tutorials/images/data_augmentation?hl=es-419), llámala $augX_2$.
  1. $augX_{1comp} \leftarrow model(augX_1)$
  1. $augX_{2comp} \leftarrow model(augX_2)$
  1. $cX_{1comp} \leftarrow cluster(augX_{1comp})$
  1. $cX_{2comp} \leftarrow cluster(augX_{2comp})$
  1. $M \leftarrow augX_{1comp} ~ augX_{2comp}^T$
  1. $loss_C \leftarrow cX_{1comp}(1 - cX_{1comp}) + cX_{2comp}(1 - cX_{2comp})$ # Puede que tengas que crear tu [propia función de coste](https://keras.io/api/losses/#creating-custom-losses)
  1. $loss_M \leftarrow crossentropy(I, softmax(M/\tau, axis=1)))$ # Puede que tengas que crear tu [propia función de coste](https://keras.io/api/losses/#creating-custom-losses)
    1. $\tau$ es un hiperparámetro que se suele definir a 5.0
  1. $loss \leftarrow loss_M + \lambda~loss_C$
    1. $\lambda$ es un hiperparámetro (puedes probar con 0.5)


In [ ]:
# Escribe aquí la solución. Crea tantos bloques de código como necesites. Puedes utilizar la siguiente red para generar distorsiones

data_augmentation = tf.keras.models.Sequential(
    [
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.RandomZoom(.15),
    ]
)

data_augmentation_2 = tf.keras.models.Sequential(
    [
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.Resizing(48, 48), # para CIFAR, para MNIST usar 40 en lugar de 48
        tf.keras.layers.RandomCrop(32, 32), # para CIFAR, para MNIST usar 28 en lugar de 32
    ]
)


# ¡ENHORABUENA! Has completado la práctica de modelos generativos.


# Trabajo extra

¿Has probado a hacer el autoencoder totalmente convolucional? Para el *decoder* puedes usar las funciones [UpSampling2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/UpSampling2D) o [Conv2DTranspose](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2DTranspose).